# ADV-05: Multi-View Geometry & 3D Computer Vision
## Epipolar Geometry, Normalized 8-Point Algorithm, DLT Triangulation & Bundle Adjustment

**Author:** Aji Pamungkas Tri Nurcahyo (Institut Teknologi Sumatera - ITERA)  
**Level:** L3 Advanced Curriculum  
**Pillar:** Pillar 1 — Intelligent Computer Vision (Roadmap ISVWS 2026–2040)  

---
### Abstract
This laboratory explores the rigorous projective geometry of multi-view 3D computer vision:
1. **Pinhole Camera Projection**: Modeling intrinsic calibration $K$ and extrinsic pose $[R \mid \mathbf{t}]$.
2. **Two-View Epipolar Geometry**: Bilinear constraint $\tilde{x}_2^T F \tilde{x}_1 = 0$ and essential matrix $E = [\mathbf{t}]_\times R$.
3. **Normalized 8-Point Algorithm**: Hartley coordinate normalization, SVD nullspace solution, and rank-2 enforcement $\det(F) = 0$.
4. **Direct Linear Transformation (DLT) Triangulation**: Solving inhomogeneous 3D point coordinates from two noisy perspective views.
5. **Bundle Adjustment**: Non-linear optimization minimizing geometric reprojection RMSE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy Version: {np.__version__}")

### 1. Synthetic 3D Scene and Calibrated Camera Rig
We define a calibrated pinhole camera rig with focal length $f = 800\text{ px}$ and baseline $b = 0.6\text{ m}$, observing a 3D wireframe cube constellation.

In [ ]:
K = np.array([
    [800.0, 0.0, 320.0],
    [0.0, 800.0, 240.0],
    [0.0, 0.0, 1.0]
], dtype=float)

# Camera 1 at origin
R1 = np.eye(3, dtype=float)
t1 = np.zeros(3, dtype=float)
P1 = K @ np.hstack([R1, t1.reshape(3, 1)])

# Camera 2: Baseline 0.6m along X, yaw 5 deg
baseline = 0.6
yaw = np.radians(5.0)
R2 = np.array([
    [np.cos(yaw), 0.0, -np.sin(yaw)],
    [0.0, 1.0, 0.0],
    [np.sin(yaw), 0.0, np.cos(yaw)]
], dtype=float)
C2 = np.array([baseline, 0.0, 0.0])
t2 = -R2 @ C2
P2 = K @ np.hstack([R2, t2.reshape(3, 1)])

# 12 3D Points (Cube + interior targets)
pts3d_clean = np.array([
    [-0.4, -0.4, 3.2], [0.4, -0.4, 3.2], [0.4, 0.4, 3.2], [-0.4, 0.4, 3.2],
    [-0.4, -0.4, 4.0], [0.4, -0.4, 4.0], [0.4, 0.4, 4.0], [-0.4, 0.4, 4.0],
    [0.0, 0.0, 3.6], [-0.2, 0.2, 4.2], [0.2, -0.2, 3.5], [0.1, 0.1, 4.5]
], dtype=float)

def project(P, K_cam, R_cam, t_cam):
    Pc = R_cam @ P + t_cam
    z = max(1e-6, Pc[2])
    u = K_cam[0, 0] * Pc[0] / z + K_cam[0, 2]
    v = K_cam[1, 1] * Pc[1] / z + K_cam[1, 2]
    return np.array([u, v])

projs1 = np.array([project(P, K, R1, t1) for P in pts3d_clean])
projs2 = np.array([project(P, K, R2, t2) for P in pts3d_clean]) + np.random.normal(0, 0.5, projs1.shape)

print(f"Projected {len(pts3d_clean)} correspondences across two views.")

### 2. Normalized 8-Point Algorithm for Fundamental Matrix $F$
We apply Hartley coordinate normalization, solve $A \mathbf{f} = 0$ via SVD, and enforce the rank-2 singularity constraint $\det(F) = 0$.

In [ ]:
def normalize_hartley(pts):
    mean = np.mean(pts, axis=0)
    s = np.sqrt(2.0) / np.mean(np.sqrt(np.sum((pts - mean)**2, axis=1)))
    T = np.array([
        [s, 0, -s * mean[0]],
        [0, s, -s * mean[1]],
        [0, 0, 1]
    ])
    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    return (T @ pts_h.T).T[:, :2], T

n1, T1 = normalize_hartley(projs1)
n2, T2 = normalize_hartley(projs2)

A = np.zeros((len(n1), 9))
for i in range(len(n1)):
    u1, v1 = n1[i]
    u2, v2 = n2[i]
    A[i] = [u2 * u1, u2 * v1, u2, v2 * u1, v2 * v1, v2, u1, v1, 1.0]

_, _, Vt = np.linalg.svd(A)
F_hat = Vt[-1].reshape(3, 3)

Uf, Sf, Vtf = np.linalg.svd(F_hat)
Sf[-1] = 0.0
F = T2.T @ (Uf @ np.diag(Sf) @ Vtf) @ T1
F /= np.linalg.norm(F)

print(f"det(F): {np.linalg.det(F):.2e} (Must be strictly ~0)")

### 3. Linear DLT Triangulation
Using known projection matrices $P_1$ and $P_2$, we reconstruct the 3D coordinates via SVD and measure metric error.

In [ ]:
def triangulate_point(p1, p2, P1_mat, P2_mat):
    A_mat = np.array([
        p1[0] * P1_mat[2] - P1_mat[0],
        p1[1] * P1_mat[2] - P1_mat[1],
        p2[0] * P2_mat[2] - P2_mat[0],
        p2[1] * P2_mat[2] - P2_mat[1]
    ])
    _, _, Vt_mat = np.linalg.svd(A_mat)
    X = Vt_mat[-1]
    return X[:3] / X[3]

triangulated = np.array([triangulate_point(p1, p2, P1, P2) for p1, p2 in zip(projs1, projs2)])
err_mm = np.mean(np.linalg.norm(triangulated - pts3d_clean, axis=1)) * 1000.0
print(f"Mean 3D DLT Reconstruction Error: {err_mm:.2f} mm")

### 4. 3D Scene Visualization
We visualize the reconstructed 3D point cloud alongside ground truth coordinates and camera optical centers.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Ground truth in red
ax.scatter(pts3d_clean[:, 0], pts3d_clean[:, 2], pts3d_clean[:, 1], color='#f87171', s=50, label='Ground Truth 3D')
# Triangulated in sky blue
ax.scatter(triangulated[:, 0], triangulated[:, 2], triangulated[:, 1], color='#38bdf8', marker='x', s=60, label='DLT Triangulated')

# Camera optical centers
ax.scatter([0], [0], [0], color='#38bdf8', s=100, label='Camera 1 (Origin)')
ax.scatter([C2[0]], [C2[2]], [C2[1]], color='#34d399', s=100, label=f'Camera 2 (b={baseline}m)')

ax.set_xlabel('X (m)')
ax.set_ylabel('Depth Z (m)')
ax.set_zlabel('Y (m)')
ax.set_title("Multi-View 3D Metric Reconstruction")
ax.legend()
plt.show()